In [ ]:
import spikeinterface.full as si
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# PATHS (set here when running via main_pipeline; standalone fallback below)
try:
    mcs_file
    base_folder
except NameError:
    mcs_file    = Path(r'C:\Users\labuser\Ilaria\C5\C5_output\2026-03-27T11-17-48McsRecording_E-00218.h5')
    base_folder = Path(r'C:\Users\labuser\Ilaria\Project\Pipeline_project\mcs_tutorial_output_3min')
base_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# MCS-SPECIFIC PARAMETERS
use_raw  = False       # False -> .h5 (Multi Channel DataManager export); True -> .raw binary
freq_min = 200.        # highpass cutoff (Hz)
freq_max = None        # bandpass upper cutoff; None -> highpass only

try:
    sorter = SORTER_MCS
except NameError:
    sorter = 'herdingspikes'

cref_operator  = 'median'
cref_reference = 'local'

# Shared variables exposed to main_pipeline
_source_path     = str(mcs_file)
_analyzer_sparse = False

In [ ]:
# LOAD
if use_raw:
    raw_rec = si.read_mcsraw(mcs_file)
else:
    raw_rec = si.read_mcsh5(mcs_file)
print(raw_rec)

In [ ]:
# PROBE SETUP
# Priority: (1) geometry embedded in file -> (2) generic rectangular grid
from probeinterface import generate_multi_columns_probe

n_ch = raw_rec.get_num_channels()

if raw_rec.get_property('contact_vector') is not None:
    print(f"Probe geometry found in file — {n_ch} channels, using as-is.")
else:
    def _best_grid(n):
        best = (1, n)
        for c in range(1, int(n ** 0.5) + 1):
            if n % c == 0:
                best = (c, n // c)
        return best

    n_cols, n_rows = _best_grid(n_ch)
    probe = generate_multi_columns_probe(
        num_columns=n_cols,
        num_contact_per_column=n_rows,
        xpitch=200,
        ypitch=200,
    )
    probe.set_device_channel_indices(np.arange(n_ch))
    raw_rec = raw_rec.set_probe(probe, group_mode='by_probe')
    print(f"No probe geometry in file — using generic {n_cols}x{n_rows} grid (200 um pitch).")

# PROBE MAP
fig, ax = plt.subplots(figsize=(5, 6))
si.plot_probe_map(raw_rec, ax=ax, with_channel_ids=True)
plt.tight_layout()
plt.show()

In [ ]:
# PREPROCESSING
_n_raw_channels = raw_rec.get_num_channels()

if manual_channel_ids is not None:
    raw_rec = raw_rec.channel_slice(manual_channel_ids)
    print(f"Manual selection: kept {len(manual_channel_ids)} channels")

rec1 = si.highpass_filter(raw_rec, freq_min=freq_min)

if artifact_timestamps_s:
    _fs       = raw_rec.get_sampling_frequency()
    _triggers = [np.array([int(t * _fs) for t in artifact_timestamps_s], dtype=np.int64)]
    rec1 = si.remove_artifacts(rec1, list_triggers=_triggers,
                                ms_before=artifact_ms_before, ms_after=artifact_ms_after,
                                mode='zeros')
    print(f"Artifact removal: blanked {len(artifact_timestamps_s)} event(s)")

bad_channel_ids, channel_labels = si.detect_bad_channels(rec1)

if extra_bad_channel_ids:
    bad_channel_ids = list(set(bad_channel_ids.tolist()) | set(extra_bad_channel_ids))

rec2 = rec1.remove_channels(bad_channel_ids)
rec  = si.common_reference(rec2, operator=cref_operator, reference=cref_reference)
print(rec)

_bad_labels = []
for _ch in bad_channel_ids:
    _idx = np.where(rec1.channel_ids == _ch)[0]
    _bad_labels.append(str(channel_labels[_idx[0]]) if len(_idx) > 0 else 'manual')
with open(base_folder / 'bad_channels.csv', 'w') as _f:
    _f.write('channel_id,label\n')
    for _c, _l in zip(bad_channel_ids, _bad_labels):
        _f.write(f'{_c},{_l}\n')
print(f'Bad channels ({len(bad_channel_ids)}) saved -> {base_folder / "bad_channels.csv"}')